In [1]:
import scipy.io
import time
import numpy as np

In [2]:
## Please do not forget to keep `movies_data.mat` file in the same folder as this notebook.

In [3]:
from tqdm import tqdm


def my_recommender(rate_mat, lr, with_reg):
    """
    :param rate_mat: training rating matrix (with zeros for missing ratings)
    :param lr: latent dimension (low rank)
    :param with_reg: boolean flag, set true for using regularization
    :return: U, V, b_u, b_i, global_bias"""

    #initializations
    n_user, n_item = rate_mat.shape
    U = np.random.rand(n_user, lr) / lr
    V = np.random.rand(n_item, lr) / lr
    b_u = np.zeros((n_user, 1))
    b_i = np.zeros((n_item, 1))
    global_bias = np.sum(rate_mat) / np.sum(rate_mat > 0)

    # TODO pick hyperparams
    max_iter = 100
    learning_rate = 5e-3
    u_reg = lr * 2e-2
    v_reg = lr * 2e-2
    b_reg = lr * 4e-2
    threshold = 1e-4

    # Pre-compute things
    lr_const = 2 * learning_rate

    # TODO implement your code here
    u_idx, i_idx = rate_mat.nonzero()
    ratings = rate_mat[u_idx, i_idx]

    if with_reg:
        for it in tqdm(range(max_iter)):
            avg_rmse = 0
            for n in range(len(ratings)):
                u,i = u_idx[n], i_idx[n]
                error = ratings[n] - (global_bias + b_u[u] + b_i[i] + U[u].dot(V[i]))
                avg_rmse += error**2

                U[u] += lr_const * (-u_reg * U[u] + error * V[i])
                V[i] += lr_const * (-v_reg * V[i] + error * U[u])
                b_u[u] += lr_const * (-b_reg * b_u[u] + error)
                b_i[i] += lr_const * (-b_reg * b_i[i] + error)
            avg_rmse /= len(ratings)
            avg_rmse = np.sqrt(avg_rmse)

            # Stopping criterion
            if avg_rmse < threshold:
                break
    else:
        for it in tqdm(range(max_iter)):
            avg_rmse = 0
            for n in range(len(ratings)):
                u,i = u_idx[n], i_idx[n]
                error = ratings[n] - (global_bias + b_u[u] + b_i[i] + U[u].dot(V[i]))
                avg_rmse += error**2

                U[u] += lr_const * (error * V[i])
                V[i] += lr_const * (error * U[u])
                b_u[u] += lr_const * error
                b_i[i] += lr_const * error
            avg_rmse /= len(ratings)
            avg_rmse = np.sqrt(avg_rmse)

            # Stopping criterion
            if avg_rmse < threshold:
                break

    return U, V, b_u, b_i, global_bias

In [4]:
cell = scipy.io.loadmat('movies_data.mat')
rate_mat = cell['train']
test_mat = cell['test']

low_rank_ls = [1, 3, 5]
for lr in low_rank_ls:
    for reg_flag in [False, True]:
        st = time.time()
        U, V, b_u, b_i, global_bias = my_recommender(rate_mat, lr, reg_flag)
        t = time.time() - st
        
        # Compute RMSE for training set
        mask_train = (rate_mat > 0)
        train_pred = global_bias + b_u + b_i.T + U.dot(V.T)
        train_rmse = np.sqrt(np.sum(((rate_mat - train_pred) * mask_train) ** 2) / float(np.sum(mask_train)))
        
        # Compute RMSE for test set
        mask_test = (test_mat > 0)
        test_pred = global_bias + b_u + b_i.T + U.dot(V.T)
        test_rmse = np.sqrt(np.sum(((test_mat - test_pred) * mask_test) ** 2) / float(np.sum(mask_test)))
        
        print('SVD-%s-%i\t%.4f\t%.4f\t%.2f\n' % ('withReg' if reg_flag else 'noReg', lr, train_rmse, test_rmse, t))

100%|██████████| 100/100 [02:10<00:00,  1.30s/it]


SVD-noReg-1	0.8636	0.9241	130.13



100%|██████████| 100/100 [02:48<00:00,  1.69s/it]


SVD-withReg-1	0.8633	0.9205	168.77



100%|██████████| 100/100 [02:07<00:00,  1.28s/it]


...

SVD-withReg-3	0.8169	0.9174	171.70



100%|██████████| 100/100 [02:07<00:00,  1.28s/it]


SVD-noReg-5	0.7550	0.9895	127.96



100%|██████████| 100/100 [02:51<00:00,  1.72s/it]

SVD-withReg-5	0.8000	0.9076	171.64

